# Folding a Tiny Piece of mRNA — Classical vs. Quantum

### A simple story, one classical example, one quantum example

You already know **superposition**, **entanglement**, **interference**, and **measurement**.
This notebook does not introduce anything new about quantum mechanics — it just shows you,
with the smallest possible example, how those four ideas let a quantum computer search for
an answer *differently* than a normal computer does.

No QAOA, no big optimization loop, no 80-qubit research pipeline. Just:

1. A tiny story (4 possible ways one short mRNA piece can fold)
2. A classical Python solution (try every option, one at a time)
3. A quantum Qiskit solution (put every option in superposition, use interference to make the
   right answer stand out, then measure)
4. Plots of the probability distribution so you can *see* the difference


## 1. The Story

A strand of mRNA is made of four kinds of bases: **A, U, G, C**. When it folds back on itself,
some bases pair up. The rule we'll use (a simplified version of the real chemistry):

| Pair | Allowed? | "Reward" for forming it |
|---|---|---|
| G — C | ✅ yes | −1 (favorable, lowers energy) |
| A — U | ✅ yes | −1 (favorable, lowers energy) |
| anything else | ❌ no | not allowed |

**Our tiny mRNA strand:** `G G C C` (positions 1, 2, 3, 4)

Only two pairings are even chemically possible here:

- **Pair A** = position 1 with position 4 → `G`-`C` ✅ allowed
- **Pair B** = position 2 with position 3 → `G`-`C` ✅ allowed

Each pair can independently be **formed** or **not formed**. That's it — that's the whole puzzle.
Two independent yes/no choices = **4 possible foldings**:

| Pair A formed? | Pair B formed? | Folding |
|---|---|---|
| No  | No  | strand stays open |
| No  | Yes | only the middle folds |
| Yes | No  | only the ends fold |
| Yes | Yes | fully folded — both pairs form |

**The question:** which of these 4 foldings is the most stable (lowest total energy)?

Common sense already tells you the answer — forming *both* pairs should be best, since each pair
you form makes the energy go down by 1. Let's confirm that two ways: classically, then quantum-mechanically.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Two independent yes/no choices -> represent as 2 bits: (pair_A, pair_B)
# 1 = pair is formed, 0 = pair is not formed
options = [(0,0), (0,1), (1,0), (1,1)]
labels  = ["00\n(open)", "01\n(B only)", "10\n(A only)", "11\n(both)"]

print("Options and their bits:")
for opt, lab in zip(options, labels):
    print(f"  {opt}  ->  {lab}")


## 2. The Classical Approach — try every option, one at a time

This is the most honest way to describe what a normal computer does: **loop through every
possibility, score it, remember the best one so far.** With only 4 options this is instant.


In [ ]:
def energy(pair_A_formed, pair_B_formed):
    """Each formed pair lowers the energy by 1. Lower energy = more stable."""
    e = 0
    if pair_A_formed:
        e -= 1
    if pair_B_formed:
        e -= 1
    return e

print(f"{'Pair A':>7} {'Pair B':>7} {'Energy':>7}")
energies = []
best_so_far = None
for (a, b) in options:
    e = energy(a, b)
    energies.append(e)
    print(f"{a:>7} {b:>7} {e:>7}")
    if best_so_far is None or e < best_so_far[0]:
        best_so_far = (e, (a, b))

print()
print(f"Classical answer -> best folding is {best_so_far[1]}, energy = {best_so_far[0]}")


In [ ]:
# A quick picture of the energy landscape we just searched by hand
plt.figure(figsize=(5,3.5))
plt.bar(labels, energies, color=["#b0b0b0", "#8fbcdb", "#8fbcdb", "#2e6f9e"])
plt.ylabel("Energy (lower = more stable)")
plt.title("Classical brute-force result: checking all 4 foldings")
plt.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()


## 3. Why This Gets Hard for a Real Computer (not for this tiny example — for a *real* mRNA strand)

For our 4-base toy strand, checking every option was nothing — 4 options, done instantly.

But real mRNA strands used in vaccines are 60+ bases long, with dozens of candidate pairs.
**Every extra yes/no choice doubles the number of possible foldings**, because each new choice
multiplies the option count by 2 (formed or not formed, independent of everything else already decided).

That's what "the number of options grows exponentially" actually means — nothing more mysterious
than repeated doubling. Let's just look at how fast that gets out of hand:


In [ ]:
num_choices = np.arange(2, 41, 2)           # number of yes/no pairing choices
num_options = 2.0 ** num_choices             # options double each time

plt.figure(figsize=(6,3.5))
plt.plot(num_choices, num_options, marker="o", color="#2e6f9e")
plt.yscale("log")
plt.xlabel("Number of yes/no pairing choices in the strand")
plt.ylabel("Number of possible foldings (log scale)")
plt.title("Why classical brute force stops being practical")
plt.axvline(2, color="gray", linestyle="--", linewidth=1)
plt.text(2.3, 10, "our toy example\n(2 choices, 4 options)", fontsize=8)
plt.tight_layout()
plt.show()

print(f"Our toy strand:  2 choices  -> {2**2} options (instant)")
print(f"A real mRNA strand: 40 choices -> {2**40:,} options (impossible to brute-force)")


**This is the actual limit** — not that a classical computer "can't fold RNA," but that
*checking every option one at a time* becomes impossible long before you reach real vaccine-sized
sequences. That's the opening for a fundamentally different search strategy: a quantum computer.


## 4. The Quantum Approach — search all 4 options *at once*, then use interference

Here's the plan, using exactly the ideas you already know:

1. **Superposition** — put both qubits into an equal superposition of all 4 possible foldings
   (`00`, `01`, `10`, `11`) at the same time, using Hadamard gates.
2. **Interference (the "oracle" + "diffusion" trick)** — flip the *phase* of the answer we know is
   best (`11`, both pairs formed), then apply a second step that turns that phase flip into a boost
   in *probability*. This is the same interference trick behind Grover's search algorithm — small,
   two-qubit version.
3. **Measurement** — collapse the superposition and see which folding we actually get. If the
   interference trick worked, `11` should come up far more often than the other three.

No optimization loop, no classical-quantum back-and-forth — just one circuit, run many times.


In [ ]:
# If Qiskit is not installed, uncomment the next line and run once:
# %pip install qiskit qiskit-aer matplotlib

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)

# Step 1: Superposition -- both qubits explore both options (formed / not formed) at once
qc.h(0)   # qubit 0 = "is pair A formed?"
qc.h(1)   # qubit 1 = "is pair B formed?"

qc.barrier()

# Step 2a: The oracle -- mark the answer we already know is best, |11>,
# by flipping its phase (no visible change to probabilities yet, just phase)
qc.cz(0, 1)

qc.barrier()

# Step 2b: The diffusion step -- turns that phase flip into a probability boost
qc.h([0, 1])
qc.x([0, 1])
qc.cz(0, 1)
qc.x([0, 1])
qc.h([0, 1])

qc.barrier()

# Step 3: Measurement
qc.measure([0, 1], [0, 1])

print(qc.draw("text"))


In [ ]:
sim = AerSimulator()
shots = 2000
result = sim.run(qc, shots=shots).result()
counts = result.get_counts()

# Qiskit reports each outcome as a string "c1c0": the LEFT character is qubit 1 (pair_B),
# the RIGHT character is qubit 0 (pair_A). Decode it so it lines up with the classical table above.
decoded_counts = {}
for bitstring, n in counts.items():
    pair_B, pair_A = int(bitstring[0]), int(bitstring[1])
    decoded_counts[(pair_A, pair_B)] = n

probs = [decoded_counts.get(opt, 0) / shots for opt in options]

print("Measured outcomes over", shots, "runs:")
for opt, lab, p in zip(options, labels, probs):
    n = decoded_counts.get(opt, 0)
    print(f"  pair_A={opt[0]} pair_B={opt[1]}  ({lab.splitlines()[0]})  ->  {n:>4} times  ({p:.1%})")


In [ ]:
plt.figure(figsize=(5,3.5))
plt.bar(labels, probs, color=["#b0b0b0", "#8fbcdb", "#8fbcdb", "#2e6f9e"])
plt.ylabel("Probability of measuring this folding")
plt.ylim(0, 1)
plt.title("Quantum result: interference boosts the best folding")
plt.tight_layout()
plt.show()


## 5. Comparing the Two Answers

- **Classical:** checked all 4 foldings one at a time, found `11` (both pairs formed) has the
  lowest energy.
- **Quantum:** started with all 4 foldings in superposition, used one round of interference, and
  measurement now shows `11` far more often than the other three outcomes.

Both agree — for this tiny example, that's expected, and it's a nice sanity check that the quantum
circuit is doing what we think it's doing.

**What actually happened inside the circuit, in plain language:**

- After the two `H` gates, all 4 foldings had *equal* probability — the computer knew nothing yet.
- The oracle step didn't change any probabilities by itself — it only flipped the *phase* of `11`,
  which you can't see directly (this is the subtle, non-classical part of the trick).
- The diffusion step is what converts that invisible phase flip into something you *can* see: it
  makes the waves representing the 4 options interfere — the flipped one constructively reinforces
  itself, and the other three destructively cancel down. That's **interference** doing real work.
- Measurement then collapses everything into one concrete outcome, sampled according to those
  now-unequal probabilities.

**Why this matters at scale:** in this 2-choice example, checking all 4 options classically was
just as easy as the quantum version — there was nothing to gain yet. The whole point of Section 3's
doubling chart is that classical brute force stops being an option once you have dozens of
choices instead of 2. A quantum circuit built the same way — superposition over *all* the choices,
then interference tuned to boost good answers — doesn't need to check foldings one at a time.
That's the entire reason this approach is interesting for real, 60-base mRNA strands, even though
this toy example was too small to show a speed difference.


## 6. Summary

1. **The story:** a 4-base mRNA snippet, 2 independent yes/no pairing choices, 4 possible foldings.
2. **Classical:** looped through all 4 foldings, scored each one, found the best by direct comparison.
3. **Why it gets hard:** every extra yes/no choice doubles the number of foldings — fine at 2 choices,
   impossible to brute-force at 40+.
4. **Quantum:** built a 2-qubit circuit — superposition to hold all options at once, then one
   interference step (oracle + diffusion) to boost the best one — and measured the result many times.
5. **Same answer, different search strategy** — and the quantum strategy is the one that doesn't
   require checking options one at a time, which is exactly what's needed once the strand (and the
   number of choices) gets big.

That's the whole idea. Everything else — QAOA, 80-qubit hardware, real 60-base sequences — is just
this same story, scaled up.
